# EXPT NO: 9
## Recommendation Systems using Collaborative Filtering Techniques
**Target Variable:** Recommended movies for a given user / Recommended items similar to a given item
**Input Feature:** User ID, Movie ID, Ratings, Timestamp

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

### SCENARIO 1: USER-BASED COLLABORATIVE FILTERING
Load data, Create User-Item matrix, Compute User Similarity

In [ ]:
# 1. Load the MovieLens dataset
columns = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('ml-100k/u.data', sep='\t', names=columns)

df.head()

In [ ]:
# 2. Create User-Item matrix & Handle missing values
user_item_matrix = df.pivot(index='user_id', columns='item_id', values='rating').fillna(0)
print("User-Item Matrix Shape:", user_item_matrix.shape)

# 3. Compute similarity between users (Cosine similarity)
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

# Heatmap for subset of User Similarity Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(user_similarity_df.iloc[:20, :20], cmap='viridis')
plt.title('User Similarity Heatmap (First 20 Users)')
plt.show()

In [ ]:
# 4. Identify top-N similar users & Predict ratings
def predict_user_based(user_item_matrix, similarity_matrix):
    pred = similarity_matrix.dot(user_item_matrix) / np.array([np.abs(similarity_matrix).sum(axis=1)]).T
    return pred

user_prediction = predict_user_based(user_item_matrix.values, user_similarity)
user_prediction_df = pd.DataFrame(user_prediction, index=user_item_matrix.index, columns=user_item_matrix.columns)

# 5. Recommend top-N movies for a user (e.g., User 1)
user_id = 1
unseen_movies = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] == 0].index
recommendations = user_prediction_df.loc[user_id, unseen_movies].sort_values(ascending=False).head(5)
print(f"\nTop 5 Recommendations for User {user_id}:")
print(recommendations)

# Evaluate using RMSE and MAE
actual_ratings = user_item_matrix.values[user_item_matrix.values.nonzero()]
predicted_ratings = user_prediction[user_item_matrix.values.nonzero()]

rmse = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
mae = mean_absolute_error(actual_ratings, predicted_ratings)
print(f"\nUser-based RMSE: {rmse:.4f}")
print(f"User-based MAE: {mae:.4f}")

### SCENARIO 2: ITEM-BASED COLLABORATIVE FILTERING
Create Item-User matrix, Compute Item Similarity

In [ ]:
# 1. Create Item-User matrix (Transpose of User-Item matrix)
item_user_matrix = user_item_matrix.T

# 2. Compute Item Similarity (Cosine similarity)
item_similarity = cosine_similarity(item_user_matrix)
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)

# Heatmap for subset of Item Similarity Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(item_similarity_df.iloc[:20, :20], cmap='coolwarm')
plt.title('Item Similarity Heatmap (First 20 Items)')
plt.show()

In [ ]:
# 3. Predict ratings for Item-based CF
def predict_item_based(user_item_matrix, similarity_matrix):
    pred = user_item_matrix.dot(similarity_matrix) / np.array([np.abs(similarity_matrix).sum(axis=1)])
    return pred

item_prediction = predict_item_based(user_item_matrix.values, item_similarity)
item_prediction_df = pd.DataFrame(item_prediction, index=user_item_matrix.index, columns=user_item_matrix.columns)

actual_ratings = user_item_matrix.values[user_item_matrix.values.nonzero()]
predicted_ratings_item = item_prediction[user_item_matrix.values.nonzero()]

rmse_item = np.sqrt(mean_squared_error(actual_ratings, predicted_ratings_item))
mae_item = mean_absolute_error(actual_ratings, predicted_ratings_item)
print(f"Item-based RMSE: {rmse_item:.4f}")
print(f"Item-based MAE: {mae_item:.4f}")

# Recommendation comparison chart
models = ['User-Based', 'Item-Based']
rmses = [rmse, rmse_item]

plt.bar(models, rmses, color=['blue', 'orange'])
plt.ylabel('RMSE')
plt.title('Comparison of Recommendation Accuracies (RMSE)')
plt.show()